# 04 — Survival & Transition Modeling

Implement Kaplan-Meier, Cox PH (competing risks: default vs prepayment), and discrete-time hazard model. Show event curves and compare against baselines.

In [ ]:
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter, CoxPHFitter

train_df = pd.read_csv('../data/synthetic/loan_monthly_performance_train.csv')
servicer_df = pd.read_csv('../data/synthetic/servicer_updates.csv')
for col in ['reporting_month','origination_month']:
    if col in train_df.columns:
        train_df[col] = pd.to_datetime(train_df[col]).dt.to_period('M')
from src.pipeline.loader import reconcile_servicer_updates
train_df, _ = reconcile_servicer_updates(train_df, servicer_df)
print(f'Train: {train_df.shape}')


## Build Survival Dataset

In [ ]:
from src.modeling.survival import SurvivalDataBuilder
builder = SurvivalDataBuilder()
survival_df = builder.build_survival_dataset(train_df)
print(f'Survival dataset: {survival_df.shape}')
print(survival_df['event_type'].value_counts().to_string())


## Kaplan-Meier Overall Survival Curve

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(survival_df['duration'], event_observed=survival_df['event'], label='All loans')
fig, ax = plt.subplots(figsize=(8, 4))
kmf.plot_survival_function(ax=ax, color='steelblue', ci_show=True)
ax.set_xlabel('Months Since Origination')
ax.set_ylabel('Survival Probability')
ax.set_title('Kaplan-Meier — Overall Loan Survival')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/km_curve.png', dpi=150)
plt.show()


## KM by Credit Score Band

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue','orange','green','red','purple']
for i, band in enumerate(survival_df['credit_score_band'].dropna().unique()[:5]):
    sub = survival_df[survival_df['credit_score_band'] == band]
    if len(sub) > 20:
        kmf2 = KaplanMeierFitter()
        kmf2.fit(sub['duration'], event_observed=sub['event'], label=str(band))
        kmf2.plot_survival_function(ax=ax, color=colors[i % len(colors)], ci_show=False)
ax.set_title('KM Survival by Credit Score Band')
ax.set_xlabel('Months')
ax.set_ylabel('Survival Probability')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/km_by_credit_band.png', dpi=150)
plt.show()


## Load Trained Survival Models

In [ ]:
results = json.load(open('../reports/survival/survival_results.json'))
rows = []
for name, res in results.items():
    m = res.get('metrics', {})
    rows.append({'model': name,
                 'concordance_index': round(m.get('concordance_index', float('nan')), 4),
                 'brier_score': round(m.get('brier_score', float('nan')), 4)})
pd.DataFrame(rows)


## Discrete-Time Hazard Model (Top Feature Importances)

In [ ]:
dt_result = results.get('discrete_time', {})
fi = dt_result.get('feature_importance', {})
if fi:
    fi_df = pd.DataFrame(list(fi.items()), columns=['feature','importance']).sort_values('importance', ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(8, 4))
    fi_df.set_index('feature')['importance'].plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Discrete-Time Hazard Model — Feature Importance')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig('../reports/discrete_time_fi.png', dpi=150)
    plt.show()
else:
    print('Feature importance not available — run survival training first.')
